> **Public release copy.** Set `<DATA_DIR>` in the CONFIG cell to the folder holding the large input files (see README / DATA availability). Internal validation cells have been removed. The small curation TSVs are included in this folder.


## CONFIG -- edit paths here
All filesystem paths live in this one block. For the public release, swap the real
`/hpc/...` paths for placeholders; nothing else in the notebook changes.

In [ ]:
library(tidyverse)

# -- CONFIG -- edit paths here -----------------------------------------------
workingpath  <- getwd()
outpath0     <- "<DATA_DIR>"
outpath      <- file.path(outpath0, "RNAquarium_outputs")
outpathvirus <- file.path(outpath, "virus_outputs")

# Inputs in outpathvirus (processed taxonomy from the metatranscriptomic pipeline)
oct_taxonomy_file <- "taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_mostrecent.tsv"
oct_withseq_file  <- "taxonomy_hits_viruses_nonphage_withsequenceandclusters_mostrecent.tsv"

# Inputs in workingpath (minimal curation)
cluster_curation_file <- "cluster_curation.tsv"           # ACCEPT: clusterLCA -> clusterLCA_curated -> Category
rejected_clusters_file<- "rejected_clusters.tsv"          # REJECT (whole cluster): non-fish clusterLCAs
rejected_taxnames_file<- "rejected_taxnames.tsv"          # REJECT (within cluster): off-target taxname_lca_NTorNR
split_overrides_file  <- "contig_split_overrides.tsv"     # contig_name -> clusterLCA_curated (per-contig splits)
# (Carp edema rescue contigs are GENERATED by rule from the input taxonomy -- no input file needed)

min_contigs_review <- 5  # a cluster is a "candidate" worth curating if it has >= this many contigs

# Outputs (written to workingpath)
out_curated_contigs  <- "curated_contigs.tsv"
out_curated_clusters <- "curated_clusters.tsv"
out_curated_fasta    <- "curated.fasta"
out_crosswalk        <- "sept_oct_crosswalk.tsv"
out_needs_review     <- "cluster_curation_NEEDS_REVIEW.tsv"
# ----------------------------------------------------------------------------

## 1. Load input taxonomy + minimal curation

`cluster_curation.tsv` is the single substantive curation file: each accepted fish-associated
`clusterLCA`, its curated name, and Category. The **presence** of a clusterLCA in this file is
what makes it "accepted" -- there is no separate filter list.

In [ ]:
cluster_curation  <- read_tsv(file.path(workingpath, cluster_curation_file), show_col_types = FALSE)
split_overrides   <- read_tsv(file.path(workingpath, split_overrides_file),  show_col_types = FALSE)
rejected_taxnames <- read_tsv(file.path(workingpath, rejected_taxnames_file), show_col_types = FALSE)

cat("Curation: ", nrow(cluster_curation), "accepted clusters,",
    n_distinct(cluster_curation$clusterLCA_curated), "curated viruses |",
    nrow(split_overrides), "split overrides |",
    nrow(rejected_taxnames), "within-cluster reject taxnames\n")
cluster_curation %>% count(Category)

oct     <- read_tsv(file.path(outpathvirus, oct_taxonomy_file), show_col_types = FALSE)
oct_seq <- read_tsv(file.path(outpathvirus, oct_withseq_file),  show_col_types = FALSE)
cat("October taxonomy:", nrow(oct), "rows | with-seq:", nrow(oct_seq), "rows\n")

## 2. Baseline cleanup (deterministic, no judgment)

The non-curation filters applied inline. Adapter rows are dropped EXCEPT the
Carp edema rescue (next cell), so we defer the Adapter drop.

In [ ]:
oct <- oct %>%
  dplyr::filter(taxname_lca_NTorNR != "Viruses",
                taxname_lca_NTorNR != "Bacteriophage sp.",
                viruscategorysimple_NTorNR != "Phage")
cat("After Viruses/Bacteriophage/Phage drop:", nrow(oct), "rows\n")

## 3. Carp edema rescue (generated by rule -- no input file)

4 Carp edema contigs were NR-flagged "Adapter" (NR hit keyword *Porcine bastrovirus*), which
would drop them with the other Adapter rows. They are recovered by rule from
a separate hand-built file -- but that recovery is *itself* generated
by a one-line filter. So we apply that rule here against the input taxonomy
(no input file): keep Adapter rows whose **NR** LCA is Carp edema virus, relabel them so they
survive the Adapter drop and join the poxvirus cluster.

In [ ]:
# The rescue rule:
#   filter(viruscategorysimple_NTorNR == "Adapter" & taxname_lca_NR == "Carp edema virus")
rescue <- oct %>%
  dplyr::filter(viruscategorysimple_NTorNR == "Adapter",
                taxname_lca_NR == "Carp edema virus") %>%
  mutate(viruscategorysimple_NTorNR = "Non-phage",
         viruscategory_NTorNR       = "Carp edema virus",
         clusterLCA                 = "Carp edema virus")  # so it joins the poxvirus cluster
cat("Carp edema rescue contigs (expect 4):", nrow(rescue), "\n")

# Drop the remaining Adapter rows, then bind the rescued ones back
oct <- oct %>% dplyr::filter(viruscategorysimple_NTorNR != "Adapter")
oct <- bind_rows(oct, rescue)
cat("After Adapter drop + rescue:", nrow(oct), "rows\n")

## 4. Filter to accepted clusters + assign curated name (the generated curation)

This regenerates the curated `clusterLCA -> curated` mapping automatically:
inner-join the input taxonomy to `cluster_curation.tsv` **on `clusterLCA`**. This both filters
to fish-associated (only curated clusters survive) and assigns `clusterLCA_curated` + `Category`.
Then apply the per-contig split overrides on the run-stable `contig_name`.

In [ ]:
curated <- oct %>%
  inner_join(cluster_curation, by = "clusterLCA")

cat("Curated contigs (after cluster filter+label):", nrow(curated),
    "| curated viruses:", n_distinct(curated$clusterLCA_curated), "\n")

# Within-cluster taxname reject : some contigs land in an accepted
# cluster but carry an off-target taxname_lca (Monkeypox, SARS-CoV-2, Caudoviricetes, Pandoravirus,
# etc.). The original pipeline dropped these AFTER cluster selection. Cluster-filtering alone is
# more permissive, so without this we keep ~68 extra contigs.
n_before <- nrow(curated)
curated <- curated %>%
  dplyr::filter(!taxname_lca_NTorNR %in% rejected_taxnames$taxname_lca_NTorNR)
cat("Dropped off-target taxname contigs (within accepted clusters):", n_before - nrow(curated), "\n")

# Deterministic contig-name exclusions:
#   - NTtarget_  : the Sprivivirus reference target genomes (not assembled contigs; NOT in the
#                  previously-released curated output)
#   - Iltovirus  : a contaminant flagged for removal
n_before <- nrow(curated)
curated <- curated %>%
  dplyr::filter(!grepl("NTtarget", contig_withLCA_withcluster),
                !grepl("Iltovirus", contig_withLCA_withcluster))
cat("Dropped NTtarget/Iltovirus contigs:", n_before - nrow(curated),
    "| remaining:", nrow(curated), "\n")

# Per-contig split overrides (e.g. one Barramundi contig -> Zebrafish neurinoma calicivirus)
curated <- curated %>%
  left_join(split_overrides %>% rename(curated_override = clusterLCA_curated,
                                       category_override = Category),
            by = "contig_name") %>%
  mutate(clusterLCA_curated = coalesce(curated_override, clusterLCA_curated),
         Category           = coalesce(category_override, Category)) %>%
  select(-curated_override, -category_override)

cat("Curated viruses after split overrides:", n_distinct(curated$clusterLCA_curated), "\n")

## 5. Self-check -- surface only GENUINELY NEW clusters

Every cluster in the input taxonomy is in one of three states:
- **accepted** -- in `cluster_curation.tsv` (these become the fish-associated set, step 4),
- **rejected** -- in `rejected_clusters.tsv` (considered and excluded as non-fish: SARS-CoV-2,
  HIV-1, plant/insect viruses, etc. -- the bulk of the virome),
- **new** -- in neither file.

A genuinely *new* cluster (>= `min_contigs_review` contigs) is the only thing worth a look on a
fresh data freeze. It is written to `_NEEDS_REVIEW` as `tbd`; you then move each row into either
`cluster_curation.tsv` (if fish-associated) or `rejected_clusters.tsv` (if not).

**First run:** `rejected_clusters.tsv` won't exist yet, so 01 *bootstraps* it -- writing all
non-accepted candidate clusters to it (this is your reject list, pre-populated from the data).
Sanity-check that file once; thereafter the review list should be short/empty.

In [ ]:
candidate_clusters <- oct %>%
  count(clusterLCA, name = "n_contigs") %>%
  dplyr::filter(n_contigs >= min_contigs_review)

rej_path <- file.path(workingpath, rejected_clusters_file)

if (!file.exists(rej_path)) {
  # BOOTSTRAP: first run -- everything not accepted becomes the initial reject list
  rejected <- candidate_clusters %>%
    anti_join(cluster_curation, by = "clusterLCA") %>%
    arrange(desc(n_contigs))
  write_tsv(rejected %>% select(clusterLCA, n_contigs), rej_path)
  cat("BOOTSTRAP: no", rejected_clusters_file, "found. Wrote initial reject list of",
      nrow(rejected), "non-fish clusters (>=", min_contigs_review, "contigs).\n")
  cat("  -> Review it ONCE to confirm none are actually fish-associated, then re-run.\n")
  rejected_clusters <- rejected
} else {
  rejected_clusters <- read_tsv(rej_path, show_col_types = FALSE)
  cat("Loaded reject list:", nrow(rejected_clusters), "clusters.\n")
}

# Genuinely new = candidate cluster in NEITHER accept nor reject list
new_clusters <- candidate_clusters %>%
  anti_join(cluster_curation,  by = "clusterLCA") %>%
  anti_join(rejected_clusters, by = "clusterLCA") %>%
  arrange(desc(n_contigs)) %>%
  mutate(clusterLCA_curated = "tbd", Category = "tbd")

if (nrow(new_clusters) > 0) {
  cat("\nNOTE:", nrow(new_clusters),
      "NEW clusters are in neither accept nor reject list. Wrote:", out_needs_review, "\n")
  cat("  -> Move each into cluster_curation.tsv (fish) or rejected_clusters.tsv (non-fish).\n")
  write_tsv(new_clusters %>% select(clusterLCA, n_contigs, clusterLCA_curated, Category),
            file.path(workingpath, out_needs_review))
  print(head(new_clusters, 20))
} else {
  cat("No new clusters -- accept + reject lists cover all candidates.\n")
}

## 6. Recompute per-curated-virus counts + classification string

Curated-cluster counts and the `;`-joined taxonomy classification used by the metacoder tree
(03) and portal (04). Genus-level curated viruses (Chuvivirus, Deltainfluenzavirus) are not
appended as an extra tip.

In [ ]:
genus_level_curated <- c("Chuvivirus", "Deltainfluenzavirus")

curated <- curated %>%
  group_by(clusterLCA_curated) %>%
  mutate(clusterLCAcurated_contigcount    = n(),
         clusterLCAcurated_bioprojectcount = n_distinct(bioproject, na.rm = TRUE),
         clusterLCAcurated_clustercount    = n_distinct(cluster, na.rm = TRUE)) %>%
  ungroup()

build_classification <- function(sk, cl, kg, ph, cls, ord, fam, gen, curated_name) {
  parts <- c(sk, cl, kg, ph, cls, ord, fam, gen)
  if (!(curated_name %in% genus_level_curated)) parts <- c(parts, curated_name)
  paste(parts[!is.na(parts) & parts != ""], collapse = ";")
}

curated <- curated %>%
  rowwise() %>%
  mutate(classification = build_classification(
    tax_superkingdom_NTorNR, tax_clade_NTorNR, tax_kingdom_NTorNR, tax_phylum_NTorNR,
    tax_class_NTorNR, tax_order_NTorNR, tax_family_NTorNR, tax_genus_NTorNR,
    clusterLCA_curated)) %>%
  ungroup()

best_cls <- curated %>%
  count(clusterLCA_curated, classification) %>%
  group_by(clusterLCA_curated) %>% slice_max(n, n = 1, with_ties = FALSE) %>% ungroup() %>%
  select(clusterLCA_curated, classification)
curated <- curated %>% select(-classification) %>%
  left_join(best_cls, by = "clusterLCA_curated")

cat("Built classification strings\n")

## 7. Sept/Oct crosswalk (keyed on contig core ID)

The September Salmon run and October taxonomy cluster slightly differently, so the *same*
contig has a different `clustersize`/`CLUSTERk` in its name. The contig **core ID**
(`PRJ..._CONTIG_N`, before the first `|`) is run-invariant; 02b joins the Salmon matrix
columns to this crosswalk on it, replacing scattered per-name string patches.
(`NTtarget_` reference genomes are dropped in step 4, so `targets` should be 0 here -- the
carve-out below is retained only as a guard in case the upstream filter ever changes.)

In [ ]:
core_id <- function(x) str_extract(x, "^[^|]+")

crosswalk <- curated %>%
  transmute(
    contig_core   = if_else(str_starts(contig_withLCA_withcluster, "NTtarget_"),
                            contig_withLCA_withcluster, core_id(contig_withLCA_withcluster)),
    oct_full_name = contig_withLCA_withcluster,
    is_target     = str_starts(contig_withLCA_withcluster, "NTtarget_")
  ) %>%
  distinct()
cat("Crosswalk entries:", nrow(crosswalk), "| targets:", sum(crosswalk$is_target), "\n")

## 8. Assemble outputs & write

Output tags (`# ->`) mark where each file is consumed, for the public-release prune.

In [ ]:
curated_out <- curated %>%
  relocate(clusterLCA_curated, .before = contig_withLCA) %>%
  relocate(Category, .before = contig_withLCA) %>%
  relocate(classification, .after = tax_species_NTorNR) %>%
  arrange(Category, clusterLCA_curated)

# -> 02a (contig list for matrix subset), 02b, 03, 04   [also = portal TSV in 04]
write_tsv(curated_out, file.path(workingpath, out_curated_contigs))
cat("Wrote", out_curated_contigs, ":", nrow(curated_out), "rows\n")

curated_clusters <- curated_out %>%
  group_by(clusterLCA_curated) %>%
  summarise(Category        = first(Category),
            classification  = first(classification),
            contigcount     = first(clusterLCAcurated_contigcount),
            bioprojectcount = first(clusterLCAcurated_bioprojectcount),
            clustercount    = first(clusterLCAcurated_clustercount),
            .groups = "drop") %>%
  arrange(Category, desc(contigcount))
# -> 03 (metacoder tree node data), 04 (portal JSON)
write_tsv(curated_clusters, file.path(workingpath, out_curated_clusters))
cat("Wrote", out_curated_clusters, ":", nrow(curated_clusters), "curated viruses\n")

# -> 02b (Sept/Oct reconciliation)
write_tsv(crosswalk, file.path(workingpath, out_crosswalk))
cat("Wrote", out_crosswalk, ":", nrow(crosswalk), "entries\n")

In [ ]:
# Fasta of curated contigs, headers as in the portal: >query=<contig_name>;taxname=<curated name>
# Most sequences come from the nonphage with-sequence file, joined on contig_withLCA_withcluster.
seq_lookup <- oct_seq %>% select(contig_withLCA_withcluster, seq.text)

# --- Rescue-contig sequences (the 4 Carp edema contigs) -------------------------------------
# These contigs are tagged Adapter/phage upstream, so they are ABSENT from the *nonphage*
# with-sequence file (oct_withseq_file). Their sequences are preserved in a tiny
# rescue_sequences.tsv, keyed on the run-stable contig_name.
#
# ALTERNATIVE SOURCE (instead of rescue_sequences.tsv): these 4 sequences DO appear in the
# masked/trimmed FASTA of the full Salmon contig set:
#     taxonomy_hits_viruses_withsequenceandclusters_notargetsforsalmon_2025-09-17_sprivireplacediwth2genomes_masked_trimmed.fa
# To source them from there instead, read that FASTA and pull the 4 records by contig_withLCA_withcluster
# (which is the FASTA header), e.g. with Biostrings:
#     library(Biostrings)
#     allseq <- readDNAStringSet(file.path(outpathvirus,
#       "taxonomy_hits_viruses_withsequenceandclusters_notargetsforsalmon_2025-09-17_sprivireplacediwth2genomes_masked_trimmed.fa"))
#     rescue_ids <- curated_out %>% dplyr::filter(clusterLCA == "Carp edema virus",
#                     !contig_withLCA_withcluster %in% oct_seq$contig_withLCA_withcluster) %>%
#                   pull(contig_withLCA_withcluster)
#     rescue_seq <- tibble(contig_withLCA_withcluster = rescue_ids,
#                          rescue_seq = as.character(allseq[rescue_ids]))  # join on contig_withLCA_withcluster instead
# Note that .fa is masked/trimmed, so sequences may differ slightly from the originals in
# rescue_sequences.tsv -- prefer the .tsv for exact reproduction of the previously-released curated fasta.
# -- default path: read the preserved 4-row file:
rescue_seq <- read_tsv(file.path(workingpath, "rescue_sequences.tsv"), show_col_types = FALSE) %>%
  select(contig_name, rescue_seq = seq.text)

fasta_df <- curated_out %>%
  left_join(seq_lookup, by = "contig_withLCA_withcluster") %>%
  left_join(rescue_seq, by = "contig_name") %>%
  mutate(seq.text = coalesce(seq.text, rescue_seq)) %>%   # fill the 4 rescue seqs
  dplyr::filter(!is.na(seq.text)) %>%
  mutate(header = paste0(">query=", contig_name, ";taxname=", clusterLCA_curated))

n_missing <- curated_out %>%
  left_join(seq_lookup, by = "contig_withLCA_withcluster") %>%
  left_join(rescue_seq, by = "contig_name") %>%
  dplyr::filter(is.na(coalesce(seq.text, rescue_seq))) %>% nrow()
cat("Contigs still missing a sequence (expect 0):", n_missing, "\n")

fasta_lines <- paste0(fasta_df$header, "\n", fasta_df$seq.text)
# -> 04 (portal fasta)
writeLines(fasta_lines, file.path(workingpath, out_curated_fasta))
cat("Wrote", out_curated_fasta, ":", nrow(fasta_df), "sequences\n")